# Granite 4.2 3B — chat, thinking, and tools on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/granite-4.2-3b-chat-tools.ipynb)

[`ibm-granite/granite-4.2-3b`](https://huggingface.co/ibm-granite/granite-4.2-3b) is IBM Granite's compact **text** reasoning model (~3.66B parameters, Apache 2.0). It writes a chain of thought inside `<think>...</think>` and then the answer, and it can call tools in the card's `<tool_call>` / `<function=…>` template. Thinking is a chat-template flag (`enable_thinking=True` by default; `False` skips the trace; `low_effort=True` shortens it). This is a text assistant notebook, not speech.

The [Granite Speech 5.0 2B notebook](https://github.com/donvito/notebooks/blob/main/colab/granite-speech-5.0-2b-asr-translation.ipynb) is a different checkpoint (`ibm-granite/granite-speech-5.0-2b`) for ASR, keyword biasing, and speech translation. It does not chat, think, or call tools.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about 15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. The card's example loads `torch.bfloat16`; this notebook uses **float16** (~7–8 GB of weights) with **SDPA** (eager if SDPA refuses the stack). 4-bit bitsandbytes is the fallback if float16 OOMs. Native context is 128K (extendable to 512K); a T4 KV cache cannot hold that, so generations stay short.

**Links:** [model card](https://huggingface.co/ibm-granite/granite-4.2-3b) · [blog](https://huggingface.co/blog/ibm-granite/granite-4-2) · [IBM docs](https://www.ibm.com/granite/docs/models/granite4-2) · [GitHub](https://github.com/ibm-granite/granite-4.2-language-models) · [collection](https://huggingface.co/collections/ibm-granite/granite-42-language-models)


In [ ]:
%pip install -q "transformers==5.20.1" accelerate bitsandbytes

# The card installs `accelerate` and `transformers` and loads GraniteForCausalLM.
# 5.20.1 is the pin used by the other T4 transformers notebooks in this repo.
# bitsandbytes is only used if float16 does not fit.



> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 1. Check the T4 and pick a dtype

T4 is `sm_75`: float16, SDPA / eager, no FlashAttention-2. The next cell stops if you are on CPU.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("4.57.1"), (
    f"transformers {transformers.__version__} is loaded; granite-4.2-3b needs a current "
    "transformers with Jinja chat templates. Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")



## 2. Load Granite 4.2 3B

`AutoTokenizer` + `AutoModelForCausalLM`, as in the [Transformers quick start](https://huggingface.co/ibm-granite/granite-4.2-3b#quick-start-transformers). First run downloads about 7.4 GB of safetensors. Attention is **sdpa**, then **eager**. If float16 does not fit, the cell retries in **4-bit**.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "ibm-granite/granite-4.2-3b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_causal(attn, quant=None):
    kwargs = {
        "attn_implementation": attn,
        "device_map": "auto",
    }
    if quant is None:
        kwargs["dtype"] = DTYPE
    else:
        kwargs["quantization_config"] = quant
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).eval()


QUANT = None
ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_causal(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    print("retrying 4-bit bitsandbytes...")
    QUANT = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    for attn in ("sdpa", "eager"):
        try:
            model = load_causal(attn, quant=QUANT)
            ATTN = attn
            break
        except Exception as exc:
            last_error = exc
            print(f"{attn} 4-bit load failed:", exc)
            del model
            model = None
            torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load granite-4.2-3b") from last_error

DEVICE = next(model.parameters()).device
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("attention:", ATTN, "| dtype:", getattr(model, "dtype", "quantized"))
print("quantization:", "4-bit nf4" if QUANT else "none (float16/bfloat16)")
print(f"parameters: {n_params:.2f}B")
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)



## 3. Generation helper

Card sampling is required for every mode: `temperature=1.0`, `top_p=0.95`, `do_sample=True`. Thinking mode on the card uses `max_new_tokens=8192`; that is too much KV for a short T4 demo, so the helper defaults to **1024** (256 when thinking is off). Pass `enable_thinking=True` / `False` and `low_effort=True` through to `apply_chat_template`.


In [ ]:
import json
import re

# Card sampling. Keep these next to generate() so later cells reuse them.
SAMPLER = dict(temperature=1.0, top_p=0.95, do_sample=True)


def generate(
    messages,
    tools=None,
    enable_thinking=True,
    low_effort=False,
    max_new_tokens=1024,
):
    # One Granite turn. Returns decoded new tokens, including think / tool tags.
    kwargs = dict(
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
        low_effort=low_effort,
    )
    if tools is not None:
        kwargs["tools"] = tools
    text = tokenizer.apply_chat_template(messages, **kwargs)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            **SAMPLER,
        )
    raw = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=False,
    )
    if torch.cuda.is_available():
        print(
            f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
            f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
        )
    return raw


def parse_model_output(text):
    # Split <think>...</think> from the final answer. Same idea as the card helper.
    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        answer_start = text.find("</think>") + len("</think>")
        answer_end = text.find("<|im_end|>", answer_start)
        answer = (
            text[answer_start:answer_end].strip()
            if answer_end != -1
            else text[answer_start:].strip()
        )
    else:
        thinking, answer = "", text.replace("<|im_end|>", "").strip()
    return thinking, answer


def parse_tool_calls(text):
    # Parse the card <tool_call><function=name><parameter=k>v</parameter> template.
    calls = []
    for match in re.finditer(
        r"<tool_call>\s*<function=([^>]+)>(.*?)</function>\s*</tool_call>",
        text,
        flags=re.DOTALL,
    ):
        name = match.group(1).strip()
        args = {}
        for pname, pvalue in re.findall(
            r"<parameter=([^>]+)>\s*(.*?)\s*</parameter>",
            match.group(2),
            flags=re.DOTALL,
        ):
            raw_value = pvalue.strip()
            try:
                args[pname] = json.loads(raw_value)
            except json.JSONDecodeError:
                args[pname] = raw_value
        calls.append({"name": name, "arguments": args})
    return calls



## 4. Demo 1 — short chat with thinking

`enable_thinking=True` (the template default) opens `<|im_start|>assistant\n<think>\n` so the model reasons before answering. The first prompt is the card's strawberry count. The second call sets `enable_thinking=False` so you can see the empty `<think></think>` shortcut.


In [ ]:
print("=== thinking (enable_thinking=True) ===")
thinking_messages = [
    {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
]
thinking_raw = generate(thinking_messages, enable_thinking=True, max_new_tokens=1024)
print(thinking_raw)
think, answer = parse_model_output(thinking_raw)
print()
print("--- think ---")
print(think or "(empty)")
print()
print("--- answer ---")
print(answer)

print()
print("=== non-thinking (enable_thinking=False) ===")
direct_messages = [
    {"role": "user", "content": "What is the capital of France?"},
]
direct_raw = generate(direct_messages, enable_thinking=False, max_new_tokens=256)
print(direct_raw)
_, direct_answer = parse_model_output(direct_raw)
print()
print("--- answer ---")
print(direct_answer)



## 5. Demo 2 — tool-calling loop

Tools use the [OpenAI function schema](https://platform.openai.com/docs/guides/function-calling) from the card. The chat template injects them under `# Tools` and asks the model to reply with `<tool_call>` / `<function=…>` only. Three mock tools: weather, local time, and a tiny FX table. The loop is the card's multi-turn recipe: assistant `tool_calls` plus `role="tool"` results go back through `apply_chat_template`.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a specified city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Name of the city"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_time_in_city",
            "description": "Get the current local time for a specified city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Name of the city"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount from one currency to another.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "ISO code, e.g. USD"},
                    "to_currency": {"type": "string", "description": "ISO code, e.g. EUR"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
]


def get_current_weather(city: str) -> dict:
    catalog = {
        "boston": {"temperature": "72°F", "condition": "Partly cloudy", "humidity": "65%"},
        "paris": {"temperature": "18°C", "condition": "Light rain", "humidity": "80%"},
    }
    return catalog.get(city.strip().lower(), {"temperature": "21°C", "condition": "Clear", "humidity": "50%", "city": city})


def get_time_in_city(city: str) -> dict:
    catalog = {
        "boston": {"timezone": "America/New_York", "local_time": "2026-10-26 10:15"},
        "paris": {"timezone": "Europe/Paris", "local_time": "2026-10-26 16:15"},
    }
    hit = catalog.get(city.strip().lower(), {"timezone": "UTC", "local_time": "2026-10-26 14:15"})
    return {"city": city, **hit}


def convert_currency(amount, from_currency: str, to_currency: str) -> dict:
    rates = {("USD", "EUR"): 0.86, ("EUR", "USD"): 1.16, ("USD", "GBP"): 0.74}
    pair = (str(from_currency).upper(), str(to_currency).upper())
    rate = rates.get(pair)
    if rate is None:
        return {"error": f"Unsupported currency pair: {pair[0]} to {pair[1]}"}
    return {
        "amount": amount,
        "from_currency": pair[0],
        "to_currency": pair[1],
        "converted_amount": round(float(amount) * rate, 2),
        "rate": rate,
    }


TOOL_FUNCTIONS = {
    "get_current_weather": get_current_weather,
    "get_time_in_city": get_time_in_city,
    "convert_currency": convert_currency,
}


def run_tool(name, arguments):
    function = TOOL_FUNCTIONS.get(name)
    if function is None:
        return {"error": f"Unknown tool: {name}"}
    try:
        return function(**arguments)
    except (TypeError, ValueError) as exc:
        return {"error": f"Bad arguments for {name}: {exc}"}


def run_tool_loop(question, max_rounds=4, max_new_tokens=1024):
    messages = [{"role": "user", "content": question}]
    print(f"[user] {question}")
    for round_number in range(1, max_rounds + 1):
        raw = generate(
            messages,
            tools=TOOLS,
            enable_thinking=True,
            max_new_tokens=max_new_tokens,
        )
        print(f"\n=== model (round {round_number}) ===")
        print(raw)
        thinking, answer = parse_model_output(raw)
        calls = parse_tool_calls(raw)
        if not calls:
            print()
            print("--- final answer ---")
            print(answer)
            return answer

        assistant = {
            "role": "assistant",
            "content": f"<think>\n{thinking}\n</think>" if thinking else "",
            "tool_calls": [
                {"function": {"name": call["name"], "arguments": call["arguments"]}}
                for call in calls
            ],
        }
        messages.append(assistant)
        for call in calls:
            result = run_tool(call["name"], call["arguments"])
            print(f"[tool call] {call['name']}({call['arguments']})")
            print(f"[tool result] {json.dumps(result)}")
            messages.append({"role": "tool", "content": json.dumps(result)})
    raise RuntimeError(f"No final answer after {max_rounds} tool-calling rounds")



In [ ]:
run_tool_loop("What's the weather like in Boston right now?")
print()
run_tool_loop("How much is 80 USD in EUR, and what time is it in Paris?")



## 6. Caveats

**Dtype.** The checkpoint is stored as bfloat16. A free T4 has no native bf16, so this notebook loads **float16** (~7–8 GB). GPUs with `sm_80+` (L4, A100) use bfloat16. Do not request bf16 on a T4.

**Attention.** No FlashAttention-2 on `sm_75`. The load cell uses **sdpa**, then **eager**. Flash SDP is turned off.

**Context length on T4.** The model natively supports 128K (long-context extension to 512K). KV cache on a 15 GB card cannot hold that window on top of 7–8 GB of weights. Keep prompts short and `max_new_tokens` in the 256–1024 range here. The card's 8192-token thinking budget is for larger GPUs.

**4-bit fallback.** If float16 OOMs, the load cell retries with bitsandbytes NF4. Quality drops a little; the chat template and tool format do not change.

**Sampling.** The card is explicit: `temperature=1.0` and `top_p=0.95` for chat, thinking, and tool calling. Do not switch to greedy for these demos.

**Apache 2.0.** Unrestricted commercial and research use. Thinking traces can contain unpolished intermediate steps; treat `<think>` as scratch work, not the answer.

**Not the Granite Speech notebook.** [`colab/granite-speech-5.0-2b-asr-translation.ipynb`](https://github.com/donvito/notebooks/blob/main/colab/granite-speech-5.0-2b-asr-translation.ipynb) loads `ibm-granite/granite-speech-5.0-2b` with `AutoModelForSpeechSeq2Seq` for ASR, keyword biasing, and X↔En speech translation. This notebook loads `ibm-granite/granite-4.2-3b` as a **text** chat / thinking / tool-calling assistant. No audio, no `AutoProcessor`, no speech prompts.

**Tools are mocks.** Weather, clocks, and FX rates are hardcoded so the loop runs offline. Swap the function bodies for real APIs when you take this out of Colab.
